# Analisis chad

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Agregar la raíz del proyecto al path para importar desde src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.auxiliar_functions import (
    RAW_DATA_DIR,
    DATA_DIR,
    SEMILLA,
    huella,
    verificar_checksums,
)

# 1. Verificar integridad SHA-256 de los 3 archivos Excel crudos
verificar_checksums()

Verificación de integridad completada: 3 archivo(s) coinciden.


True

In [2]:
# Rutas de los 3 archivos Excel en data/raw/
ruta_2020_2022 = RAW_DATA_DIR / "Totalizadores Planta 2020_2022.xlsx"
ruta_2021_2023 = RAW_DATA_DIR / "Totalizadores Planta - 2021_2023.xlsx"
ruta_2022_2023 = RAW_DATA_DIR / "Totalizadores Planta 2022_2023.xlsx"

# Consultamos los nombres de las hojas de cada archivo con pandas
hojas_2020_2022 = pd.ExcelFile(ruta_2020_2022).sheet_names
hojas_2021_2023 = pd.ExcelFile(ruta_2021_2023).sheet_names
hojas_2022_2023 = pd.ExcelFile(ruta_2022_2023).sheet_names

print("Cantidad de hojas en 2020_2022:", len(hojas_2020_2022))
print("Cantidad de hojas en 2021_2023:", len(hojas_2021_2023))
print("Cantidad de hojas en 2022_2023:", len(hojas_2022_2023))

print("\nLista de hojas en 2020_2022:")
print(hojas_2020_2022)

# Vemos que hoja tiene de mas el archivo 2021_2023 respecto a los otros dos
hoja_diferente = set(hojas_2021_2023) - set(hojas_2020_2022)
print("\nHoja que solo existe en 2021_2023:", hoja_diferente)

Cantidad de hojas en 2020_2022: 15
Cantidad de hojas en 2021_2023: 16
Cantidad de hojas en 2022_2023: 15

Lista de hojas en 2020_2022:
['Consolidado KPI', 'Consolidado Produccion', 'Totalizadores Produccion', 'Consolidado EE', 'Totalizadores Energia', 'Consolidado Agua', 'Totalizadores Agua', 'Consolidado GasVapor', 'Totalizadores Gas y Vapor', 'Consolidado Aire', 'Totalizadores Aire', 'Totalizadores CO2', 'Totalizadores Efluentes', 'Totalizadores Glicol', 'Auxiliar']

Hoja que solo existe en 2021_2023: {'Seguimiento Dia'}


In [3]:
import pandas as pd
import numpy as np

# 1. Clasificación de hojas según naturaleza física
HOJAS_NO_ACUMULATIVAS = {"Consolidado KPI", "Auxiliar", "Seguimiento Dia"}
HORAS_CANONICAS = [f"{h:02d}:00" for h in range(24)] + ["23:59"]

# 2. Contenedores globales de trazabilidad
log_filas_eliminadas = []   # Registra 23:59:59 y bloques clonados de fin de mes
log_reparaciones = []       # Registra 1900-01-01, typos horarios y picos 01:00 tolerados
log_dias_eliminados = []    # Lista final de fechas (DIA) descartadas y su motivo exacto

In [4]:
def limpiar_y_auditar_hoja(df_raw, archivo, hoja):
    df = df_raw.copy()
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]

    if "DIA" not in df.columns or "HORA" not in df.columns:
        return pd.DataFrame()

    # Eliminar filas sin fecha ni hora
    df = df.dropna(subset=["DIA", "HORA"]).copy()
    df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["DIA"]).copy()

    s_hora = df["HORA"].astype(str).str.strip()

    # A. Detectar y reparar prefijos de fecha anómalos (ej. 1900-01-01)
    mask_prefijo_raro = s_hora.str.match(r"^\d{4}-\d{2}-\d{2}\s+") & ~s_hora.str.startswith("1899-12-30")
    if mask_prefijo_raro.any():
        for d in df.loc[mask_prefijo_raro, "DIA"].unique():
            log_reparaciones.append({
                "DIA": d, "Archivo": archivo, "Hoja": hoja,
                "Tipo": "Prefijo de fecha en HORA",
                "Detalle": "Se eliminó prefijo '1900-01-01' conservando la hora"
            })

    # Normalización base de texto horario
    s_hora = s_hora.str.replace(r"^\d{4}-\d{2}-\d{2}\s+", "", regex=True)
    s_hora = s_hora.str.replace(r"\.0$", "", regex=True)
    s_hora = s_hora.str.replace(r"^(\d):", r"0\1:", regex=True)

    # B. Detectar y BORRAR registros '23:59:59'
    mask_235959 = s_hora.str.startswith("23:59:59")
    if mask_235959.any():
        for d in df.loc[mask_235959, "DIA"].unique():
            log_filas_eliminadas.append({
                "DIA": d, "Archivo": archivo, "Hoja": hoja,
                "Motivo": "Registro extra '23:59:59'",
                "Filas_Borradas": int((df["DIA"] == d).sum() and mask_235959[df["DIA"] == d].sum())
            })
        df = df[~mask_235959].copy()
        s_hora = s_hora[~mask_235959]

    # Truncar segundos residuales (:00, :01) a HH:MM
    df["HORA"] = s_hora.str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)

    # C. Detectar y BORRAR bloques horarios clonados (meses de 28 o 30 días con 50/100 filas)
    es_nuevo_bloque = (df["HORA"] == "00:00") & (df["DIA"] == df["DIA"].shift(1))
    id_bloque = es_nuevo_bloque.groupby(df["DIA"]).cumsum()
    mask_clonados = id_bloque > 0
    if mask_clonados.any():
        for d, grp in df[mask_clonados].groupby("DIA"):
            log_filas_eliminadas.append({
                "DIA": d, "Archivo": archivo, "Hoja": hoja,
                "Motivo": "Jornada apilada/clonada (mes < 31 días)",
                "Filas_Borradas": len(grp)
            })
        df = df[~mask_clonados].copy()

    # D. Reparar typos humanos de secuencia (ej. 15->15->17 o 13->16->15->16)
    horas_corregidas = []
    for d, grupo in df.groupby("DIA", sort=False):
        h_list = grupo["HORA"].tolist()
        h_orig = h_list.copy()

        if len(h_list) == 25 and h_list[0] == "00:00" and h_list[-1] == "23:59":
            if h_list != HORAS_CANONICAS:
                h_list = HORAS_CANONICAS.copy()
        else:
            for i in range(1, len(h_list) - 1):
                prev_h, curr_h, next_h = h_list[i - 1], h_list[i], h_list[i + 1]
                if prev_h in HORAS_CANONICAS[:-1] and next_h in HORAS_CANONICAS[:-1]:
                    idx_p = HORAS_CANONICAS.index(prev_h)
                    idx_n = HORAS_CANONICAS.index(next_h)
                    if idx_n - idx_p == 2 and curr_h != HORAS_CANONICAS[idx_p + 1]:
                        h_list[i] = HORAS_CANONICAS[idx_p + 1]

        if h_list != h_orig:
            cambios = [f"{o}->{n}" for o, n in zip(h_orig, h_list) if o != n]
            log_reparaciones.append({
                "DIA": d, "Archivo": archivo, "Hoja": hoja,
                "Tipo": "Typo de secuencia horaria",
                "Detalle": f"Horas corregidas por posición: {', '.join(cambios)}"
            })
        horas_corregidas.extend(h_list)

    df["HORA"] = horas_corregidas
    df = df.drop_duplicates(subset=["DIA", "HORA"], keep="first")
    return df

In [5]:
archivos_priorizados = [
    ("2022_2023", ruta_2022_2023),
    ("2021_2023", ruta_2021_2023),
    ("2020_2022", ruta_2020_2022),
]

dfs_por_archivo = []

for etiqueta, ruta in archivos_priorizados:
    xls = pd.ExcelFile(ruta)
    hojas_limpias = []

    for hoja in xls.sheet_names:
        hoja_clean = hoja.strip()
        df_raw = pd.read_excel(xls, sheet_name=hoja)
        df_h = limpiar_y_auditar_hoja(df_raw, etiqueta, hoja_clean)

        if not df_h.empty:
            renombres = {c: f"{c}_{hoja_clean}" for c in df_h.columns if c not in ["DIA", "HORA"]}
            df_h = df_h.rename(columns=renombres).set_index(["DIA", "HORA"])
            hojas_limpias.append(df_h)

    df_arch = pd.concat(hojas_limpias, axis=1, join="outer").reset_index()
    df_arch["_archivo_origen"] = etiqueta
    dfs_por_archivo.append(df_arch)

# Apilar priorizando el archivo más reciente en los solapamientos
df_apilado = pd.concat(dfs_por_archivo, ignore_index=True)
df_unificado = (
    df_apilado.drop_duplicates(subset=["DIA", "HORA"], keep="first")
    .sort_values(["DIA", "HORA"])
    .reset_index(drop=True)
)

# Convertir todas las variables (excepto claves) a numérico; textos como '#VALUE!' o '#DIV/0!' pasan a NaN
cols_datos = [c for c in df_unificado.columns if c not in ["DIA", "HORA", "_archivo_origen"]]
for c in cols_datos:
    df_unificado[c] = pd.to_numeric(df_unificado[c], errors="coerce")

print(f"Dataset unificado pre-filtrado de días: {df_unificado.shape[0]} filas | {df_unificado['DIA'].nunique()} fechas únicas.")

Dataset unificado pre-filtrado de días: 29700 filas | 1190 fechas únicas.


In [6]:
col_target = "Frio (Kw)_Consolidado EE"
cols_acumulativas = [
    c for c in cols_datos
    if not any(c.endswith(f"_{h}") for h in HOJAS_NO_ACUMULATIVAS)
]

TOL = 1e-3
dias_validos = []

for dia, grupo in df_unificado.groupby("DIA", sort=True):
    grupo = grupo.sort_values("HORA")
    horas = grupo["HORA"].tolist()
    archivo_origen = grupo["_archivo_origen"].iloc[-1]
    n_reg = len(horas)

    # --- 1. EVALUACIÓN ESTRUCTURAL ---
    tiene_00 = "00:00" in horas
    tiene_2359 = "23:59" in horas

    fallos_est = []
    if not tiene_00:
        fallos_est.append("No arranca a las 00:00 (primera hora: " + str(horas[0]) + ")")
    if not tiene_2359:
        fallos_est.append("Sin registro de cierre 23:59 (última hora: " + str(horas[-1]) + ")")
    if n_reg < 24:
        fallos_est.append(f"Cantidad de horas insuficiente ({n_reg}/25)")

    if fallos_est:
        log_dias_eliminados.append({
            "DIA": dia.strftime("%Y-%m-%d"),
            "Archivo_Origen": archivo_origen,
            "Instancia": "1. Estructural (Horas)",
            "Variable_Afectada": "HORA",
            "Motivo_Eliminacion": " | ".join(fallos_est),
            "Filas_Eliminadas": n_reg
        })
        continue

    # --- 2. EVALUACIÓN FÍSICA SOBRE EL TARGET (Frio Kw) ---
    s_frio = grupo.set_index("HORA")[col_target].dropna()
    fallos_fis = []

    if len(s_frio) == 0:
        fallos_fis.append("Día sin datos numéricos (todo NaN / vacío)")
    else:
        val_00 = s_frio.get("00:00", np.nan)
        val_01 = s_frio.get("01:00", np.nan)
        val_02 = s_frio.get("02:00", np.nan)
        val_2359 = s_frio.get("23:59", np.nan)
        diffs = s_frio.diff().dropna()

        # A. Negativos
        if (s_frio < 0).any():
            fallos_fis.append(f"Valores negativos detectados (mín: {s_frio.min():.1f} kW)")

        # B. Cierre 23:59 nulo, cero o imposible (millones)
        if pd.isna(val_2359) or val_2359 <= 0:
            fallos_fis.append(f"Valor de cierre 23:59 nulo o cero ({val_2359})")
        elif val_2359 > 100_000:
            fallos_fis.append(f"Valor de cierre 23:59 imposible/extremo ({val_2359:,.1f} kW)")

        # C. Sin reinicio a las 00:00
        if pd.notna(val_00) and ((val_00 > 5000) or (pd.notna(val_01) and val_00 > val_01 + TOL)):
            fallos_fis.append(f"Totalizador no reinició a las 00:00 (00:00={val_00:,.1f} kW)")

        # D. Monotonía y manejo del pico a la 01:00
        pico_01 = pd.notna(val_01) and pd.notna(val_02) and (val_01 > val_02 + TOL)
        es_monotono = (diffs >= -TOL).all()

        if pico_01:
            # Evaluamos si desde las 02:00 hasta las 23:59 crece normalmente
            s_desde_02 = s_frio.loc[s_frio.index >= "02:00"]
            monotono_desde_02 = (s_desde_02.diff().dropna() >= -TOL).all()
            if monotono_desde_02:
                log_reparaciones.append({
                    "DIA": dia, "Archivo": archivo_origen, "Hoja": "Consolidado EE",
                    "Tipo": "Pico 01:00 aislado (Día conservado)",
                    "Detalle": f"00:00={val_00}, 01:00={val_01}, 02:00={val_02}. Se reinicia en 02:00 y cierra bien en 23:59 ({val_2359} kW)"
                })
            else:
                fallos_fis.append(f"Pico a la 01:00 ({val_01:,.1f} kW) + caídas posteriores de monotonía")
        elif not es_monotono:
            hora_caida = diffs.idxmin()
            magnitud_caida = diffs.min()
            fallos_fis.append(f"Violación de monotonía (cae {magnitud_caida:,.1f} kW a las {hora_caida})")

    if fallos_fis:
        log_dias_eliminados.append({
            "DIA": dia.strftime("%Y-%m-%d"),
            "Archivo_Origen": archivo_origen,
            "Instancia": "2. Física (Target Acumulativo)",
            "Variable_Afectada": col_target,
            "Motivo_Eliminacion": " | ".join(fallos_fis),
            "Filas_Eliminadas": n_reg
        })
        continue

    dias_validos.append(dia)

# Filtrar el dataset limpio final
df_limpio = df_unificado[df_unificado["DIA"].isin(dias_validos)].reset_index(drop=True)

In [7]:
df_bajas_dias = pd.DataFrame(log_dias_eliminados)
df_bajas_filas = pd.DataFrame(log_filas_eliminadas)
df_reparaciones = pd.DataFrame(log_reparaciones)

print("=" * 80)
print("RESUMEN FINAL DEL PIPELINE DE LIMPIEZA")
print("=" * 80)
print(f"Días totales antes de filtrar:   {df_unificado['DIA'].nunique()}")
print(f"Días eliminados en total:        {len(df_bajas_dias)}")
print(f"Días limpios conservados:        {df_limpio['DIA'].nunique()} ({len(df_limpio)} filas)")
print("=" * 80)

print("\n1. LISTA COMPLETA DE DÍAS ELIMINADOS Y SU MOTIVO:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", None):
    display(df_bajas_dias)

print("\n2. RESUMEN DE FILAS EXTRA ELIMINADAS (23:59:59 y Bloques Clonados de Fin de Mes):")
if not df_bajas_filas.empty:
    resumen_filas = (
        df_bajas_filas.groupby(["Archivo", "Motivo"], as_index=False)
        .agg(Dias_Afectados=("DIA", "nunique"), Total_Filas_Borradas=("Filas_Borradas", "sum"))
    )
    display(resumen_filas)

print("\n3. REGISTRO DE DÍAS REPARADOS / CONSERVADOS CON ANOMALÍAS SUBSANABLES:")
if not df_reparaciones.empty:
    resumen_rep = (
        df_reparaciones.drop_duplicates(subset=["DIA", "Tipo", "Detalle"])
        .sort_values(["Tipo", "DIA"])
        .reset_index(drop=True)
    )
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", None):
        display(resumen_rep)

RESUMEN FINAL DEL PIPELINE DE LIMPIEZA
Días totales antes de filtrar:   1190
Días eliminados en total:        31
Días limpios conservados:        1159 (28971 filas)

1. LISTA COMPLETA DE DÍAS ELIMINADOS Y SU MOTIVO:


,DIA,Archivo_Origen,Instancia,Variable_Afectada,Motivo_Eliminacion,Filas_Eliminadas
0,2020-07-11,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Totalizador no reinició a las 00:00 (00:00=29,113.0 kW) | Violación de monotonía (cae -29,113.0 kW a las 01:00)",25
1,2020-09-11,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -28,674.0 kW a las 13:00)",25
2,2021-01-03,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,Violación de monotonía (cae -10.0 kW a las 18:00),25
3,2021-01-21,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,Violación de monotonía (cae -13.0 kW a las 06:00),25
4,2021-01-31,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -32,771,277,123,571,584.0 kW a las 12:00)",25
5,2021-03-18,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -27,138.0 kW a las 07:00)",25
6,2021-03-19,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -26,003.0 kW a las 13:00)",25
7,2021-04-04,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -31,131.0 kW a las 18:00)",25
8,2021-04-21,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -5,405.0 kW a las 12:00)",25
9,2021-04-22,2020_2022,2. Física (Target Acumulativo),Frio (Kw)_Consolidado EE,"Violación de monotonía (cae -32,771,276,799,998,576.0 kW a las 17:00)",25



2. RESUMEN DE FILAS EXTRA ELIMINADAS (23:59:59 y Bloques Clonados de Fin de Mes):


,Archivo,Motivo,Dias_Afectados,Total_Filas_Borradas
0,2020_2022,Jornada apilada/clonada (mes < 31 días),610,13320
1,2020_2022,Registro extra '23:59:59',2,15
2,2021_2023,Jornada apilada/clonada (mes < 31 días),8,4800
3,2021_2023,Registro extra '23:59:59',1,16
4,2022_2023,Jornada apilada/clonada (mes < 31 días),6,3000
5,2022_2023,Registro extra '23:59:59',1,15



3. REGISTRO DE DÍAS REPARADOS / CONSERVADOS CON ANOMALÍAS SUBSANABLES:


,DIA,Archivo,Hoja,Tipo,Detalle
0,2021-06-12,2020_2022,Consolidado EE,Pico 01:00 aislado (Día conservado),"00:00=0.0, 01:00=4111695.0, 02:00=3558.0. Se reinicia en 02:00 y cierra bien en 23:59 (37992.0 kW)"
1,2021-07-21,2021_2023,Consolidado Agua,Prefijo de fecha en HORA,Se eliminó prefijo '1900-01-01' conservando la hora
2,2020-07-20,2020_2022,Totalizadores CO2,Typo de secuencia horaria,Horas corregidas por posición: 23:00->11:00
3,2020-09-05,2020_2022,Consolidado KPI,Typo de secuencia horaria,Horas corregidas por posición: 15:00->16:00
4,2021-09-16,2021_2023,Seguimiento Dia,Typo de secuencia horaria,"Horas corregidas por posición: 09:00->08:00, 08:00->09:00"
5,2022-12-10,2022_2023,Consolidado KPI,Typo de secuencia horaria,Horas corregidas por posición: 16:00->14:00


## Cargo a csv

In [8]:
import pandas as pd
from pathlib import Path

# --- 1. PREPARACIÓN DEL DATAFRAME FINAL ---
# 'df_limpio' es el DataFrame resultante del Paso 4 (filtrado de días válidos)

# Eliminar la columna auxiliar de trazabilidad '_archivo_origen' antes de exportar
# ya que no es una feature del modelo.
columnas_a_eliminar = [col for col in df_limpio.columns if "_archivo_origen" in col]
df_exportar = df_limpio.drop(columns=columnas_a_eliminar, errors="ignore")

# --- 2. CONFIGURACIÓN DE RUTAS ---
# Definir la ruta de salida (asegurarse de que la carpeta 'processed' exista)
# El TP indica guardar en data/processed/dataset_final.csv o unificado
directorio_salida = Path("../data/processed")
directorio_salida.mkdir(parents=True, exist_ok=True)
ruta_csv_salida = directorio_salida / "dataset_unificado_limpio.csv"

# --- 3. EXPORTACIÓN A CSV ---
# Exportar sin índice numérico (index=False) para evitar que al reabrir 
# se cree una columna extra 'Unnamed: 0'
print(f"Iniciando exportación a: {ruta_csv_salida}")
df_exportar.to_csv(ruta_csv_salida, index=False, encoding="utf-8")

# --- 4. VERIFICACIÓN Y REPORTE ---
print("\n" + "="*50)
print("✅ EXPORTACIÓN COMPLETADA")
print("="*50)
print(f"Ruta del archivo:   {ruta_csv_salida.resolve()}")
print(f"Dimensiones (filas, columnas): {df_exportar.shape}")
print(f"Rango de fechas:    {df_exportar['DIA'].min().strftime('%Y-%m-%d')} a {df_exportar['DIA'].max().strftime('%Y-%m-%d')}")
print(f"Fechas únicas:      {df_exportar['DIA'].nunique()}")
print("="*50)

# (Opcional pero recomendado según el TP): Calcular el nuevo SHA-256
import hashlib
import json

def calcular_sha256(ruta_archivo, tamano_bloque=65536):
    sha256 = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(tamano_bloque), b""):
            sha256.update(bloque)
    return sha256.hexdigest()

ruta_checksums = Path("../data/checksums.json")
try:
    checksums = json.loads(ruta_checksums.read_text(encoding="utf-8")) if ruta_checksums.exists() else {}
    hash_calculado = calcular_sha256(ruta_csv_salida)
    checksums[ruta_csv_salida.name] = hash_calculado
    ruta_checksums.write_text(json.dumps(checksums, indent=4), encoding="utf-8")
    print(f"🔒 SHA-256 calculado y actualizado en checksums.json: {hash_calculado}")
except Exception as e:
    print(f"⚠️ No se pudo actualizar checksums.json: {e}")

Iniciando exportación a: ..\data\processed\dataset_unificado_limpio.csv

✅ EXPORTACIÓN COMPLETADA
Ruta del archivo:   D:\existencia\facu cosa\tp_final_datos_2\data\processed\dataset_unificado_limpio.csv
Dimensiones (filas, columnas): (28971, 421)
Rango de fechas:    2020-07-01 a 2023-10-25
Fechas únicas:      1159
🔒 SHA-256 calculado y actualizado en checksums.json: 452e1f5041e52312f6189dbbfa9a43425e0c9846492c0979f5db0a61f5e89074
